# Solar Filament Segmentation - Version 21 offline fixed

No pip/network dependency. Uses the exact `FallbackUNet2D` architecture used by `best_solar_unet.pth`. Writes official compressed RLE columns and skips empty masks.

In [ ]:
import glob, os
from pathlib import Path
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_SIZE=(512,512); OUTPUT_SIZE=(2048,2048); THRESHOLD=0.50
BATCH_SIZE=8; NUM_WORKERS=2; OUTPUT_PATH='/kaggle/working/submission_v21_offline_fixed.csv'
def find_dir():
    for pattern in ['/kaggle/input/**/MAGFiLO_1.0_Kaggle_2026/test/test_images','/kaggle/input/**/test/test_images']:
        found=sorted(glob.glob(pattern,recursive=True))
        if found: return found[0]
    raise FileNotFoundError('No test/test_images directory found under /kaggle/input.')
TEST_IMG_DIR=find_dir()
CHECKPOINTS=sorted(glob.glob('/kaggle/input/**/best_solar_unet.pth',recursive=True)+glob.glob('/kaggle/input/**/best_solar_unet.zip',recursive=True)+glob.glob('/kaggle/input/**/best_solar_unet.pt',recursive=True))
if not CHECKPOINTS: raise FileNotFoundError('Attach best_solar_unet.pth or best_solar_unet.zip as a Kaggle input.')
print({'device':str(DEVICE),'test_dir':TEST_IMG_DIR,'checkpoints':CHECKPOINTS})

In [ ]:
def conv_block(in_c,out_c):
    return nn.Sequential(nn.Conv2d(in_c,out_c,3,padding=1,bias=False),nn.BatchNorm2d(out_c),nn.ReLU(inplace=True),nn.Conv2d(out_c,out_c,3,padding=1,bias=False),nn.BatchNorm2d(out_c),nn.ReLU(inplace=True))
class FallbackUNet2D(nn.Module):
    def __init__(self,in_channels=3,out_channels=1):
        super().__init__(); self.enc1=conv_block(in_channels,64); self.enc2=conv_block(64,128); self.enc3=conv_block(128,256); self.enc4=conv_block(256,512); self.pool=nn.MaxPool2d(2,2); self.bottleneck=conv_block(512,1024); self.up4=nn.ConvTranspose2d(1024,512,2,2); self.dec4=conv_block(1024,512); self.up3=nn.ConvTranspose2d(512,256,2,2); self.dec3=conv_block(512,256); self.up2=nn.ConvTranspose2d(256,128,2,2); self.dec2=conv_block(256,128); self.up1=nn.ConvTranspose2d(128,64,2,2); self.dec1=conv_block(128,64); self.final_conv=nn.Conv2d(64,out_channels,1)
    def forward(self,x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1)); e3=self.enc3(self.pool(e2)); e4=self.enc4(self.pool(e3)); b=self.bottleneck(self.pool(e4)); d4=self.dec4(torch.cat([self.up4(b),e4],1)); d3=self.dec3(torch.cat([self.up3(d4),e3],1)); d2=self.dec2(torch.cat([self.up2(d3),e2],1)); d1=self.dec1(torch.cat([self.up1(d2),e1],1)); return self.final_conv(d1)
def load_model(path):
    model=FallbackUNet2D().to(DEVICE); state=torch.load(path,map_location='cpu')
    if isinstance(state,dict) and 'state_dict' in state: state=state['state_dict']
    if isinstance(state,dict) and 'model_state_dict' in state: state=state['model_state_dict']
    state={k.removeprefix('module.').removeprefix('model.'):v for k,v in state.items()}
    missing,unexpected=model.load_state_dict(state,strict=False)
    if missing or unexpected: raise RuntimeError(f'Checkpoint mismatch: missing={missing[:3]}, unexpected={unexpected[:3]}')
    return model.eval()
models=[load_model(p) for p in CHECKPOINTS]; print(f'Loaded {len(models)} checkpoint(s).')

In [ ]:
class TestDataset(Dataset):
    def __init__(self,directory):
        exts=('*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG'); self.paths=sorted(p for e in exts for p in glob.glob(os.path.join(directory,e)))
        if not self.paths: raise FileNotFoundError(f'No test images in {directory}')
    def __len__(self): return len(self.paths)
    def __getitem__(self,i):
        p=self.paths[i]; image=cv2.imread(p,cv2.IMREAD_COLOR)
        if image is None: raise ValueError(f'Cannot read {p}')
        image=cv2.cvtColor(image,cv2.COLOR_BGR2RGB); image=cv2.resize(image,INPUT_SIZE,interpolation=cv2.INTER_AREA).astype(np.float32)/255.0
        image=(image-np.array([.485,.456,.406],np.float32))/np.array([.229,.224,.225],np.float32)
        return torch.from_numpy(image.transpose(2,0,1)).float(),os.path.basename(p)
@torch.inference_mode()
def predict(loader):
    names=[]; masks=[]
    for images,batch_names in tqdm(loader,desc='Inference'):
        probs=torch.stack([torch.sigmoid(m(images.to(DEVICE))) for m in models]).mean(0)[:,0].cpu().numpy()
        for name,prob in zip(batch_names,probs): names.append(name); masks.append((cv2.resize(prob,OUTPUT_SIZE,interpolation=cv2.INTER_LINEAR)>=THRESHOLD).astype(np.uint8))
    return names,masks
loader=DataLoader(TestDataset(TEST_IMG_DIR),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'))
image_names,predicted_masks=predict(loader); print(f'Predicted {len(predicted_masks)} masks.')

In [ ]:
def coco_compress_counts(counts):
    chars=[]
    for i,value in enumerate(counts):
        value=int(value); value-=int(counts[i-2]) if i>2 else 0; value=((-value)<<1|1) if value<0 else value<<1
        while value>=0x20: chars.append(chr((value&0x1f)|0x20|48)); value>>=5
        chars.append(chr(value+48))
    return ''.join(chars)
def encode_coco_rle(mask):
    mask=np.asarray(mask,dtype=np.uint8)
    if mask.shape!=OUTPUT_SIZE: raise ValueError(f'Expected {OUTPUT_SIZE}, got {mask.shape}')
    if not np.any(mask): return None
    pixels=np.asfortranarray(mask).reshape(-1,order='F'); changes=np.flatnonzero(np.diff(np.concatenate(([0],pixels,[0])))); counts=[]; previous=0
    for start,end in zip(changes[0::2],changes[1::2]): counts.extend([int(start-previous),int(end-start)]); previous=int(end)
    if previous<pixels.size: counts.append(int(pixels.size-previous))
    return coco_compress_counts(counts)
def build_submission(names,masks,path):
    rows=[]
    for name,mask in zip(names,masks):
        rle=encode_coco_rle(mask)
        if rle is not None and rle.strip(): rows.append({'filament_id':f'{Path(name).stem}_1','segmentation_rle':rle})
    frame=pd.DataFrame(rows,columns=['filament_id','segmentation_rle'])
    if frame.empty or frame.isna().any().any() or frame['segmentation_rle'].astype(str).str.strip().eq('').any() or frame['filament_id'].duplicated().any(): raise ValueError('Invalid submission: empty, null, blank RLE, or duplicate ID.')
    frame.to_csv(path,index=False,lineterminator='\n'); return frame
submission=build_submission(image_names,predicted_masks,OUTPUT_PATH); print(f'Wrote {len(submission)} rows to {OUTPUT_PATH}')

In [ ]:
checked=pd.read_csv(OUTPUT_PATH,keep_default_na=False)
assert checked.columns.tolist()==['filament_id','segmentation_rle']
assert not checked.empty and not checked.isna().any().any()
assert not checked['filament_id'].astype(str).str.strip().eq('').any()
assert not checked['segmentation_rle'].astype(str).str.strip().eq('').any()
assert not checked['filament_id'].duplicated().any()
print('VALIDATED:',OUTPUT_PATH); print(checked.head())